# 02 – Vorverarbeitung 

hier werden die Beschwerdetexte der Stichprobe in  Wortlisten umgewandelt, die später vektorisiert und  ausgewertet werden.

**Eingabe:** 
Stichprobe von 20.000 Beschwerden, gezogen aus den 366.945 Beschwerden mit Text ohne Duplikate

**Ausgabe:** 
pro Beschwerde eine Liste bereinigter Wörter 

**Schritte:**
1. Platzhalter wie „XXXX" und Datumsmuster mit regulären Ausdrücken entfernen
2. Texte mit spaCy (`en_core_web_sm`) in Wörter zerlegen und auf Grundformen bringen
3. Filtern: nur Nomen, Verben und Adjektive ohne Stoppwörter, mindestens drei Buchstaben
4. Auffälligkeiten prüfen und dokumentieren (z. B. fehlerhafte Grundformen)

In [ ]:
import re
import pandas as pd
import spacy
from collections import Counter

In [ ]:
import pandas as pd

sample = pd.read_csv("data/sample_20000.csv")
col = "Consumer complaint narrative"
print(sample.shape)
sample.head(3)

In [ ]:
import re

def clean_text(text):
    text = re.sub(r"X{2,}(/X{2,})*(/\d{2,4})?", " ", text)  # XXXX, XX/XX/2018
    text = re.sub(r"\{\$[\d.,]+\}", " ", text)               # Beträge wie {$100.00}
    text = re.sub(r"\s+", " ", text).strip()                  # mehrfache Leerzeichen
    return text

beispiel = sample[col].iloc[0]
print(beispiel[:300])
print("---")
print(clean_text(beispiel)[:300])

In [ ]:
sample["text_clean"] = sample[col].apply(clean_text)

print("Texte mit XX vorher:", sample[col].str.contains("XX").sum())
print("Texte mit XX nachher:", sample["text_clean"].str.contains("XX").sum())

In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm")

doc = nlp(sample["text_clean"].iloc[0][:300])

for token in doc:
    print(f"{token.text:15} {token.lemma_:15} {token.pos_:8} Stoppwort: {token.is_stop}")

In [ ]:
ALLOWED_POS = {"NOUN", "VERB", "ADJ"}

def extract_tokens(doc):
    return [
        token.lemma_.lower()
        for token in doc
        if token.is_alpha
        and not token.is_stop
        and token.pos_ in ALLOWED_POS
        and len(token.lemma_) > 2
    ]

print(extract_tokens(doc))

## 2.4 Filterregeln 

**Ablauf:** 
Zuerst entferne ich Platzhalter mit Regex. Danach zerlegt spaCy die Texte in Wörter und bildet den Stamm. Erst am Ende schreibe ich  klein, weil spaCy Wortarten am Originaltext zuverlässiger erkennt.

**Behalten werden** 
Wörter, die
- nur aus Buchstaben bestehen,
- keine Stoppwörter sind,
- Nomen, Verben oder Adjektive sind,
- mehr als zwei Buchstaben haben.

**Begründung**
- **Eigennamen verworfen:** Eigennamen sind in der Regel irrelevant und erhöhen unnötiges Rauschen
- **Adverbien verworfen:** Wörter wie „currently" tragen wenig zum Thema bei.
- **Verben behalten:** Sie tragen zur Semantik bei
- **Mindestlänge über zwei Buchstaben:** So fallen Kürzel weg, aber kurze Wörter wie „car" oder „bus" bleiben erhalten.



In [ ]:
test = sample["text_clean"].head(1000)

docs = nlp.pipe(test, disable=["parser", "ner"], batch_size=100)
tokens_test = [extract_tokens(d) for d in docs]

print(len(tokens_test))
print(tokens_test[0])
print(tokens_test[1])

## Auffälligkeit: Lemmatisierungsfehler bei „taxes"

**Problem:** 
Das Modell `en_core_web_sm` bildet aus „taxes" die Grundform „taxis" statt „tax". Dadurch zerfällt ein Begriff in zwei Wörter („tax" und „taxis"), die später getrennt gezählt werden.

**Vorgehen:** 
Ich prüfe an 1.000 Texten, ob es weitere falsche Grundformen gibt, indem ich die häufigsten Paare aus Originalwort und Grundform durchlese.

**Entscheidung**
- Bleibt es bei „taxis" und wenigen seltenen Einzelfällen, bleibe ich bei `en_core_web_sm` und korrigiere „taxis" nachträglich zu „tax".
- Tauchen mehrere häufige Fehler auf, teste ich zuerst `en_core_web_md`, danach gegebenenfalls `en_core_web_trf`.


**Anmerkung:** 
Die Korrektur ersetzt auch ein echtes „taxi" im Plural. In Finanzbeschwerden dürfte das selten sein.